In [3]:
import math
import cmath

# Решение линейного уравнения: a*x + b = 0
def solve_linear(a, b):
    if a == 0:
        return [] if b != 0 else ["Все действительные числа"]
    return [-b / a]

# Решение квадратного уравнения: a*x^2 + b*x + c = 0
def solve_quadratic(a, b, c):
    if a == 0:
        return solve_linear(b, c)

    discriminant = b**2 - 4*a*c
    if discriminant >= 0:
        sqrt_d = math.sqrt(discriminant)
    else:
        sqrt_d = cmath.sqrt(discriminant)

    root1 = (-b + sqrt_d) / (2*a)
    root2 = (-b - sqrt_d) / (2*a)

    return [root1, root2]

# Решение кубического уравнения: a*x^3 + b*x^2 + c*x + d = 0
def solve_cubic(a, b, c, d):
    if a == 0:
        return solve_quadratic(b, c, d)

    b, c, d = b/a, c/a, d/a
    p = c - b**2/3
    q = d - b*c/3 + 2*b**3/27

    discriminant = (q/2)**2 + (p/3)**3

    if discriminant == 0 and p == 0:
        root = -b/3
        return [root, root, root]
    elif discriminant >= 0:
        sqrt_d = math.sqrt(discriminant)
    else:
        sqrt_d = cmath.sqrt(discriminant)

    u = (-q/2 + sqrt_d) ** (1/3)
    v = (-q/2 - sqrt_d) ** (1/3)

    root1 = u + v - b/3
    root2 = -(u+v)/2 - b/3 + (u-v)*cmath.sqrt(3)*1j/2
    root3 = -(u+v)/2 - b/3 - (u-v)*cmath.sqrt(3)*1j/2

    return [root1, root2, root3]

# Решение уравнения четвертой степени: a*x^4 + b*x^3 + c*x^2 + d*x + e = 0
def solve_quartic(a, b, c, d, e):
    if a == 0:
        return solve_cubic(b, c, d, e)

    b, c, d, e = b/a, c/a, d/a, e/a

    p = c - 3*b**2/8
    q = d - b*c/2 + b**3/8
    r = e - b*d/4 + b**2*c/16 - 3*b**4/256

    cubic_roots = solve_cubic(1, p/2, p**2/16 - r, -q**2/64)

    for y in cubic_roots:
        if isinstance(y, complex) or y > 0:
            break
    else:
        y = max(cubic_roots, key=lambda x: x.real)

    R = math.sqrt(y) if y >= 0 else cmath.sqrt(y)
    if abs(R) < 1e-10:
        D = math.sqrt(y**2 - 4*r) if (y**2 - 4*r) >= 0 else cmath.sqrt(y**2 - 4*r)
        root1 = solve_quadratic(1, R, (y - D)/2)
        root2 = solve_quadratic(1, -R, (y + D)/2)
    else:
        D = math.sqrt(3*y**2/4 - 2*p - 2*q/R) if (3*y**2/4 - 2*p - 2*q/R) >= 0 else cmath.sqrt(3*y**2/4 - 2*p - 2*q/R)
        root1 = solve_quadratic(1, R + D, y/2 + R*D/2 - q/(2*R))
        root2 = solve_quadratic(1, -R + D, y/2 - R*D/2 + q/(2*R))

    roots = root1 + root2
    return [x - b/4 for x in roots]

# общая функция решения уравнения
def solve_polynomial(coefficients):
    coefficients = [c for c in coefficients if c != 0] or [0] # коэффициенты
    degree = len(coefficients) - 1 # coefficients[0] соответствует старшей степени

    if degree == 0:
        return ["Все действительные числа"] if coefficients[0] == 0 else []
    elif degree == 1:
        return solve_linear(coefficients[0], coefficients[1])
    elif degree == 2:
        return solve_quadratic(coefficients[0], coefficients[1], coefficients[2])
    elif degree == 3:
        return solve_cubic(coefficients[0], coefficients[1], coefficients[2], coefficients[3])
    elif degree == 4:
        return solve_quartic(coefficients[0], coefficients[1], coefficients[2], coefficients[3], coefficients[4])
    else:
        raise ValueError("Степень полинома должна быть не больше 4")

# вывод корней в формате
def format_root(root, tolerance=1e-10):
    if isinstance(root, str):
        return root

    # Обработка комплексных чисел
    if isinstance(root, complex):
        real_part = root.real
        imag_part = root.imag

        # Если мнимая часть очень мала, считаем число действительным
        if abs(imag_part) < tolerance:
            return format_root(real_part, tolerance)

        # Форматирование действительной части
        if abs(real_part) < tolerance:
            real_str = ""
        else:
            real_str = f"{real_part:.6g}".rstrip('0').rstrip('.')
            if real_str == "-0":
                real_str = ""

        # Форматирование мнимой части
        if abs(imag_part - 1) < tolerance:
            imag_str = "i"
        elif abs(imag_part + 1) < tolerance:
            imag_str = "-i"
        else:
            imag_str = f"{imag_part:.6g}".rstrip('0').rstrip('.') + "i"

        if real_str and imag_str:
            if imag_str.startswith('-'):
                return f"{real_str}{imag_str}"
            else:
                return f"{real_str}+{imag_str}"
        else:
            return real_str + imag_str

    # Обработка действительных чисел
    if abs(root) < tolerance:
        return "0"

    # Округление
    formatted = f"{root:.6g}".rstrip('0').rstrip('.')
    return formatted if formatted else "0"

# Ввод коэффициентов
def input_coefficients():
    while True:
        try:
            print("\nВыберите степень уравнения (1-4):")
            print("1. Линейное уравнение (ax + b = 0)")
            print("2. Квадратное уравнение (ax² + bx + c = 0)")
            print("3. Кубическое уравнение (ax³ + bx² + cx + d = 0)")
            print("4. Уравнение 4-й степени (ax⁴ + bx³ + cx² + dx + e = 0)")
            print("0. Выход")

            choice = input("Ваш выбор: ").strip()
            if choice == '0':
                print("Выход из программы.")
                return

            if choice not in ['1', '2', '3', '4']:
                print("Ошибка: выберите число от 0 до 4")
                continue

            degree = int(choice)
            coefficients = []
            print(f"\nВведите коэффициенты для уравнения {degree}-й степени:")

            # Ввод коэффициентов
            for i in range(degree, -1, -1):
                while True:
                    try:
                        if i == 0:
                            coef_str = input(f"Введите свободный член: ")
                        else:
                            coef_str = input(f"Введите коэффициент для x^{i}: ")

                        if '/' in coef_str:
                            parts = coef_str.split('/')
                            if len(parts) == 2:
                                numerator = float(parts[0])
                                denominator = float(parts[1])
                                if denominator == 0:
                                    print("Ошибка: деление на ноль")
                                    continue
                                value = numerator / denominator
                            else:
                                print("Ошибка: неверный формат дроби")
                                continue
                        else:
                            value = float(coef_str)

                        coefficients.append(value)
                        break
                    except ValueError:
                        print("Ошибка: введите число (можно использовать дробь вида a/b)")

            # Решение уравнения
            print(f"\nРешение уравнения: ", end="")

            # Формирование красивого уравнения для вывода
            equation_parts = []
            for i, coef in enumerate(coefficients):
                power = degree - i
                if coef == 0:
                    continue

                if power == 0:
                    equation_parts.append(f"{coef}")
                elif power == 1:
                    if coef == 1:
                        equation_parts.append("x")
                    elif coef == -1:
                        equation_parts.append("-x")
                    else:
                        equation_parts.append(f"{coef}x")
                else:
                    if coef == 1:
                        equation_parts.append(f"x^{power}")
                    elif coef == -1:
                        equation_parts.append(f"-x^{power}")
                    else:
                        equation_parts.append(f"{coef}x^{power}")

            equation = " + ".join(equation_parts).replace("+ -", "- ") + " = 0"
            print(equation)

            # Нахождение корней
            roots = solve_polynomial(coefficients)

            # Вывод результатов
            if not roots:
                print("Уравнение не имеет решений")
            elif roots == ["Все действительные числа"]:
                print("Решение: все действительные числа")
            else:
                print("Корни уравнения:")
                for i, root in enumerate(roots, 1):
                    formatted_root = format_root(root)
                    print(f"  x{i} = {formatted_root}")

            while True:
                continue_choice = input("\nРешить еще одно уравнение? (y/n): ").strip().lower()
                if continue_choice in ['y', 'н', 'да', 'yes']:
                    break
                elif continue_choice in ['n', 'т', 'нет', 'no']:
                    print("Выход из программы.")
                    return
                else:
                    print("Пожалуйста, введите 'y' или 'n'")

        except KeyboardInterrupt:
            print("\n\nПрограмма прервана пользователем.")
            return
        except Exception as e:
            print(f"Произошла ошибка: {e}")
            print("Попробуйте еще раз.")

def main():
    input_coefficients()

if __name__ == "__main__":
    main()


Выберите степень уравнения (1-4):
1. Линейное уравнение (ax + b = 0)
2. Квадратное уравнение (ax² + bx + c = 0)
3. Кубическое уравнение (ax³ + bx² + cx + d = 0)
4. Уравнение 4-й степени (ax⁴ + bx³ + cx² + dx + e = 0)
0. Выход
Ваш выбор: 1

Введите коэффициенты для уравнения 1-й степени:
Введите коэффициент для x^1: 2
Введите свободный член: 4

Решение уравнения: 2.0x + 4.0 = 0
Корни уравнения:
  x1 = -2

Решить еще одно уравнение? (y/n): 2
Пожалуйста, введите 'y' или 'n'

Решить еще одно уравнение? (y/n): y

Выберите степень уравнения (1-4):
1. Линейное уравнение (ax + b = 0)
2. Квадратное уравнение (ax² + bx + c = 0)
3. Кубическое уравнение (ax³ + bx² + cx + d = 0)
4. Уравнение 4-й степени (ax⁴ + bx³ + cx² + dx + e = 0)
0. Выход
Ваш выбор: 2

Введите коэффициенты для уравнения 2-й степени:
Введите коэффициент для x^2: 1
Введите коэффициент для x^1: 0
Введите свободный член: -4

Решение уравнения: x^2 - 4.0 = 0
Корни уравнения:
  x1 = 4

Решить еще одно уравнение? (y/n): y

Выберите с